# FieldVision — Stage 3
## Lightweight Frozen-Model Workload Profiling

**Purpose:** measure representative AI inference cost on real agricultural inputs for the NOMS workload model.

This notebook:
- does **not train** any model;
- uses a very small sample by default;
- uses frozen TorchVision models as **systems proxies**, not as final agricultural perception models;
- measures input size, decode/preprocess time, inference latency, output size, and optional CUDA memory;
- saves a compact workload profile CSV for later use in the FieldVision simulator.

### Default representative workloads

1. **Leafy crop classification** → MobileNetV3-Small
2. **Leafy context classification** → MobileNetV3-Small
3. **Rumex detection** → Faster R-CNN MobileNetV3-320-FPN
4. **Leafy orthomosaic analysis** → lightweight tiled MobileNetV3-Small proxy

These are **runtime-characterization proxies only**. Do not use their predictions as scientific results for the SenSys paper.

## 0. Optional installation

Run only if PyTorch/TorchVision are not already installed.

In [3]:
# Uncomment only if needed:
! pip install torch torchvision pandas pillow matplotlib

  Using cached torch-2.14.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (39 kB)
  Using cached torchvision-0.29.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (5.7 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.14.1-cp314-cp314-macosx_14_0_arm64.whl (127.3 MB)
Using cached torchvision-0.29.1-cp314-cp314-macosx_14_0_arm64.whl (1.8 MB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.8/875.8 kB 6.7 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [torchvision] 7/8 [torchvision]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice]

## 1. Imports and project paths

In [4]:
from pathlib import Path
import math
import time
import random
import warnings

import pandas as pd
import numpy as np
from PIL import Image

import torch
import torchvision
from torchvision.models import (
    mobilenet_v3_small,
    MobileNet_V3_Small_Weights,
)
from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_320_fpn,
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
)

HOME = Path.home()
PROJECT_ROOT = HOME / "Downloads" / "NOMS_Fieldvision"
DATASETS_DIR = PROJECT_ROOT / "datasets"
OUTPUT_DIR = PROJECT_ROOT / "eda_outputs"

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"{OUTPUT_DIR} does not exist. Run Stage 1 and Stage 2 first."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. This is fine for the lightweight default profile.")

Project root: /Users/bwbgv/Downloads/NOMS_Fieldvision
Device: cpu
Running on CPU. This is fine for the lightweight default profile.


## 2. Lightweight configuration

The defaults are intentionally small.

If everything works and you later want more stable numbers, increase the sample counts.

In [5]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Keep these small for the first run.
N_LEAFY_CROP = 20
N_LEAFY_CONTEXT = 20
N_RUMEX = 8
N_ORTHOMOSAIC = 4

# Orthomosaic profiling:
# benchmark only a few tiles, then estimate the full tiled runtime.
ORTHO_TILE_SIZE = 640
MAX_MEASURED_TILES_PER_ORTHO = 4

# If pretrained weights cannot be downloaded, the notebook will fall back
# to the exact same architecture with random weights. Runtime remains useful,
# but predictions are meaningless either way for this systems-only experiment.
USE_PRETRAINED = True

print("Configuration loaded.")

Configuration loaded.


## 3. Load Stage 1 catalogs

In [6]:
def read_catalog(name):
    path = OUTPUT_DIR / f"{name}_image_catalog.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

leafy_catalog = read_catalog("LeafySpurge")
rumex_catalog = read_catalog("RumexWeeds")

print("Leafy rows:", len(leafy_catalog))
print("Rumex rows:", len(rumex_catalog))

display(leafy_catalog.head())

Leafy rows: 1000
Rumex rows: 1000


,dataset,relative_path,filename,extension,width,height,megapixels,channels_or_mode,format,file_size_mb,error
0,LeafySpurge,data/orthomosaic/orthomosaic_30_20.tif,orthomosaic_30_20.tif,.tif,4096,4096,16.777216,RGBA,TIFF,32.861188,NaN
1,LeafySpurge,data/orthomosaic/orthomosaic_21_18.tif,orthomosaic_21_18.tif,.tif,4096,4096,16.777216,RGBA,TIFF,31.144897,NaN
2,LeafySpurge,data/crop/train/absent/854_8.tif,854_8.tif,.tif,39,39,0.001521,RGBA,TIFF,0.006172,NaN
3,LeafySpurge,data/crop/train/present/824_8.tif,824_8.tif,.tif,39,39,0.001521,RGBA,TIFF,0.006172,NaN
4,LeafySpurge,data/crop/train/present/778_7.tif,778_7.tif,.tif,39,39,0.001521,RGBA,TIFF,0.006172,NaN


## 4. Find dataset roots automatically

In [7]:
def find_dataset_folder(keyword):
    candidates = [
        p for p in DATASETS_DIR.rglob("*")
        if p.is_dir() and keyword.lower() in p.name.lower()
    ]
    if not candidates:
        return None
    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]

LEAFY_ROOT = find_dataset_folder("leafy")
RUMEX_ROOT = find_dataset_folder("rumex")

print("Leafy root:", LEAFY_ROOT)
print("Rumex root:", RUMEX_ROOT)

if LEAFY_ROOT is None or RUMEX_ROOT is None:
    raise RuntimeError("Could not locate LeafySpurge or RumexWeeds.")

Leafy root: /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets/datasets/LeafySpurge
Rumex root: /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets/datasets/RumexWeeds


## 5. Identify the four workload groups

Folder names are used only to identify crop/context/orthomosaic subsets.

In [8]:
leafy = leafy_catalog.dropna(subset=["relative_path"]).copy()
leafy["path_lower"] = leafy["relative_path"].astype(str).str.lower()

leafy_crop = leafy[leafy["path_lower"].str.contains("crop", na=False)].copy()
leafy_context = leafy[leafy["path_lower"].str.contains("context", na=False)].copy()
leafy_ortho = leafy[leafy["path_lower"].str.contains("orthomosaic", na=False)].copy()

rumex = rumex_catalog.dropna(subset=["relative_path"]).copy()

print("Leafy crop candidates:", len(leafy_crop))
print("Leafy context candidates:", len(leafy_context))
print("Leafy orthomosaic candidates:", len(leafy_ortho))
print("Rumex candidates:", len(rumex))

Leafy crop candidates: 377
Leafy context candidates: 346
Leafy orthomosaic candidates: 274
Rumex candidates: 1000


## 6. Load frozen proxy models

In [9]:
def load_classifier():
    try:
        if USE_PRETRAINED:
            weights = MobileNet_V3_Small_Weights.DEFAULT
            model = mobilenet_v3_small(weights=weights)
            transform = weights.transforms()
            pretrained = True
        else:
            raise RuntimeError("Pretrained disabled.")
    except Exception as e:
        warnings.warn(
            f"Could not load pretrained MobileNet weights ({e}). "
            "Using the same architecture with random weights for runtime profiling."
        )
        weights = None
        model = mobilenet_v3_small(weights=None)
        # Simple ImageNet-like runtime transform; scientific predictions are not used.
        transform = torchvision.transforms.Compose([
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
        ])
        pretrained = False

    model = model.to(DEVICE).eval()
    return model, transform, pretrained


def load_detector():
    try:
        if USE_PRETRAINED:
            weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
            model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights)
            transform = weights.transforms()
            pretrained = True
        else:
            raise RuntimeError("Pretrained disabled.")
    except Exception as e:
        warnings.warn(
            f"Could not load pretrained detector weights ({e}). "
            "Using the same architecture with random weights for runtime profiling."
        )
        weights = None
        model = fasterrcnn_mobilenet_v3_large_320_fpn(
            weights=None,
            weights_backbone=None,
        )
        transform = torchvision.transforms.ToTensor()
        pretrained = False

    model = model.to(DEVICE).eval()
    return model, transform, pretrained


classifier, classifier_tf, cls_pretrained = load_classifier()
detector, detector_tf, det_pretrained = load_detector()

print("Classifier pretrained:", cls_pretrained)
print("Detector pretrained:", det_pretrained)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /Users/bwbgv/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100.0%
0.2%

Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth" to /Users/bwbgv/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth


100.0%


Classifier pretrained: True
Detector pretrained: True


## 7. Utility functions

In [10]:
def model_parameter_mb(model):
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    return total_bytes / (1024 ** 2)


def tensor_output_bytes(obj):
    """Approximate tensor payload size of model output."""
    if torch.is_tensor(obj):
        return obj.numel() * obj.element_size()

    if isinstance(obj, dict):
        return sum(tensor_output_bytes(v) for v in obj.values())

    if isinstance(obj, (list, tuple)):
        return sum(tensor_output_bytes(v) for v in obj)

    return 0


def sync_if_cuda():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def cuda_memory_start():
    if DEVICE.type != "cuda":
        return None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return torch.cuda.memory_allocated()


def cuda_memory_delta_mb(start_bytes):
    if DEVICE.type != "cuda" or start_bytes is None:
        return np.nan

    peak = torch.cuda.max_memory_allocated()
    return max(0, peak - start_bytes) / (1024 ** 2)


print("Classifier parameters: %.2f MB" % model_parameter_mb(classifier))
print("Detector parameters: %.2f MB" % model_parameter_mb(detector))

Classifier parameters: 9.70 MB
Detector parameters: 73.95 MB


## 8. Classification benchmark helper

Measures:
- file size;
- image decode time;
- model preprocessing time;
- frozen inference latency;
- approximate output tensor size;
- optional additional peak CUDA memory.

In [18]:
@torch.inference_mode()
def profile_classifier_image(path, workload_name):
    file_size_mb = path.stat().st_size / (1024 ** 2)

    t0 = time.perf_counter()
    with Image.open(path) as img:
        img = img.convert("RGB")
        width, height = img.size
        t1 = time.perf_counter()

        x = classifier_tf(img)
        t2 = time.perf_counter()

    x = x.unsqueeze(0).to(DEVICE)

    mem_start = cuda_memory_start()

    sync_if_cuda()
    t3 = time.perf_counter()

    y = classifier(x)

    sync_if_cuda()
    t4 = time.perf_counter()

    return {
        "workload": workload_name,
        "model": "MobileNetV3-Small",
        "device": str(DEVICE),
        "path": str(path),
        "input_size_mb": file_size_mb,
        "width": width,
        "height": height,
        "megapixels": width * height / 1e6,
        "decode_ms": (t1 - t0) * 1000,
        "model_preprocess_ms": (t2 - t1) * 1000,
        "inference_ms": (t4 - t3) * 1000,
        "output_kb": tensor_output_bytes(y) / 1024,
        "extra_peak_cuda_mb": cuda_memory_delta_mb(mem_start),
        "tiles_measured": 1,
        "tiles_total": 1,
        "estimated_full_inference_ms": (t4 - t3) * 1000,
        "pretrained_weights": cls_pretrained,
    }

## 9. Detection benchmark helper for Rumex

In [12]:
@torch.inference_mode()
def profile_detector_image(path):
    file_size_mb = path.stat().st_size / (1024 ** 2)

    t0 = time.perf_counter()
    with Image.open(path) as img:
        img = img.convert("RGB")
        width, height = img.size
        t1 = time.perf_counter()

        x = detector_tf(img)
        t2 = time.perf_counter()

    x = x.to(DEVICE)

    mem_start = cuda_memory_start()

    sync_if_cuda()
    t3 = time.perf_counter()

    y = detector([x])

    sync_if_cuda()
    t4 = time.perf_counter()

    return {
        "workload": "Rumex weed detection",
        "model": "FasterRCNN-MobileNetV3-320-FPN",
        "device": str(DEVICE),
        "path": str(path),
        "input_size_mb": file_size_mb,
        "width": width,
        "height": height,
        "megapixels": width * height / 1e6,
        "decode_ms": (t1 - t0) * 1000,
        "model_preprocess_ms": (t2 - t1) * 1000,
        "inference_ms": (t4 - t3) * 1000,
        "output_kb": tensor_output_bytes(y) / 1024,
        "extra_peak_cuda_mb": cuda_memory_delta_mb(mem_start),
        "tiles_measured": 1,
        "tiles_total": 1,
        "estimated_full_inference_ms": (t4 - t3) * 1000,
        "pretrained_weights": det_pretrained,
    }

## 10. Lightweight orthomosaic tiling benchmark

To keep this light, the notebook does **not** process every tile.

It:
1. calculates how many 640×640 tiles the full orthomosaic would contain;
2. benchmarks at most 4 representative tiles;
3. records measured per-tile cost;
4. produces an **estimated** full tiled inference time.

Keep the `estimated_...` field separate from measured values in any paper.

In [13]:
def tile_positions(width, height, tile_size):
    xs = list(range(0, width, tile_size))
    ys = list(range(0, height, tile_size))
    return [(x, y) for y in ys for x in xs]


@torch.inference_mode()
def profile_orthomosaic(path):
    file_size_mb = path.stat().st_size / (1024 ** 2)

    t0 = time.perf_counter()
    with Image.open(path) as img:
        img = img.convert("RGB")
        width, height = img.size
        t1 = time.perf_counter()

        positions = tile_positions(width, height, ORTHO_TILE_SIZE)
        total_tiles = len(positions)

        # Spread sampled positions across the image rather than taking only top-left tiles.
        if total_tiles <= MAX_MEASURED_TILES_PER_ORTHO:
            chosen = positions
        else:
            idx = np.linspace(
                0,
                total_tiles - 1,
                MAX_MEASURED_TILES_PER_ORTHO,
                dtype=int
            )
            chosen = [positions[i] for i in idx]

        preprocess_ms = []
        inference_ms = []
        output_bytes = []
        mem_deltas = []

        for x0, y0 in chosen:
            tile = img.crop((
                x0,
                y0,
                min(x0 + ORTHO_TILE_SIZE, width),
                min(y0 + ORTHO_TILE_SIZE, height)
            ))

            p0 = time.perf_counter()
            x = classifier_tf(tile).unsqueeze(0).to(DEVICE)
            p1 = time.perf_counter()

            mem_start = cuda_memory_start()

            sync_if_cuda()
            i0 = time.perf_counter()

            y = classifier(x)

            sync_if_cuda()
            i1 = time.perf_counter()

            preprocess_ms.append((p1 - p0) * 1000)
            inference_ms.append((i1 - i0) * 1000)
            output_bytes.append(tensor_output_bytes(y))
            mem_deltas.append(cuda_memory_delta_mb(mem_start))

    median_tile_inference = float(np.median(inference_ms))

    return {
        "workload": "Leafy orthomosaic tiled analysis",
        "model": "MobileNetV3-Small tiled proxy",
        "device": str(DEVICE),
        "path": str(path),
        "input_size_mb": file_size_mb,
        "width": width,
        "height": height,
        "megapixels": width * height / 1e6,
        "decode_ms": (t1 - t0) * 1000,
        "model_preprocess_ms": float(np.sum(preprocess_ms)),
        "inference_ms": float(np.sum(inference_ms)),  # measured tiles only
        "output_kb": float(np.sum(output_bytes)) / 1024,
        "extra_peak_cuda_mb": (
            float(np.nanmax(mem_deltas))
            if DEVICE.type == "cuda"
            else np.nan
        ),
        "tiles_measured": len(chosen),
        "tiles_total": total_tiles,
        "estimated_full_inference_ms": median_tile_inference * total_tiles,
        "pretrained_weights": cls_pretrained,
    }

## 11. Run a tiny warm-up

This reduces first-inference initialization effects.

In [14]:
@torch.inference_mode()
def warmup():
    print("Warming up models...")

    x = torch.randn(1, 3, 224, 224, device=DEVICE)
    sync_if_cuda()
    _ = classifier(x)
    sync_if_cuda()

    # Detector warmup can be noticeably slower on CPU, but only runs once.
    x_det = torch.rand(3, 320, 320, device=DEVICE)
    sync_if_cuda()
    _ = detector([x_det])
    sync_if_cuda()

    print("Warm-up complete.")

warmup()

Warming up models...
Warm-up complete.


## 12. Sample real inputs

In [15]:
def sample_rows(df, n):
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=SEED).copy()


crop_sample = sample_rows(leafy_crop, N_LEAFY_CROP)
context_sample = sample_rows(leafy_context, N_LEAFY_CONTEXT)
rumex_sample = sample_rows(rumex, N_RUMEX)
ortho_sample = sample_rows(leafy_ortho, N_ORTHOMOSAIC)

print("Crop:", len(crop_sample))
print("Context:", len(context_sample))
print("Rumex:", len(rumex_sample))
print("Orthomosaic:", len(ortho_sample))

Crop: 20
Context: 20
Rumex: 8
Orthomosaic: 4


## 13. Run profiling

In [16]:
results = []

print("Profiling Leafy crop...")
for _, row in crop_sample.iterrows():
    path = LEAFY_ROOT / row["relative_path"]
    if path.exists():
        results.append(
            profile_classifier_image(
                path,
                "Leafy crop classification"
            )
        )

print("Profiling Leafy context...")
for _, row in context_sample.iterrows():
    path = LEAFY_ROOT / row["relative_path"]
    if path.exists():
        results.append(
            profile_classifier_image(
                path,
                "Leafy context classification"
            )
        )

print("Profiling Rumex detector...")
for _, row in rumex_sample.iterrows():
    path = RUMEX_ROOT / row["relative_path"]
    if path.exists():
        results.append(
            profile_detector_image(path)
        )

print("Profiling orthomosaic proxy...")
for _, row in ortho_sample.iterrows():
    path = LEAFY_ROOT / row["relative_path"]
    if path.exists():
        results.append(
            profile_orthomosaic(path)
        )

profile_df = pd.DataFrame(results)

print("\nCompleted profiles:", len(profile_df))
display(profile_df.head())

Profiling Leafy crop...
Profiling Leafy context...
Profiling Rumex detector...
Profiling orthomosaic proxy...

Completed profiles: 52


,workload,model,device,path,input_size_mb,width,height,megapixels,decode_ms,model_preprocess_ms,inference_ms,output_kb,extra_peak_cuda_mb,tiles_measured,tiles_total,estimated_full_inference_ms,pretrained_weights
0,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,18.683667,4.330042,63.927834,3.90625,NaN,1,1,63.927834,True
1,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,6.843667,1.112458,51.005167,3.90625,NaN,1,1,51.005167,True
2,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,2.393250,1.249041,49.624958,3.90625,NaN,1,1,49.624958,True
3,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,1.795750,0.873667,47.724042,3.90625,NaN,1,1,47.724042,True
4,Leafy crop classification,MobileNetV3-Small,cpu,/Users/bwbgv/Downloads/NOMS_Fieldvision/datase...,0.006172,39,39,0.001521,1.395292,1.238250,46.957459,3.90625,NaN,1,1,46.957459,True


## 14. Summarize workload profiles

In [17]:
def p95(x):
    return np.percentile(x, 95)

summary = (
    profile_df
    .groupby(["workload", "model", "device"], as_index=False)
    .agg(
        samples=("path", "count"),
        median_input_mb=("input_size_mb", "median"),
        median_megapixels=("megapixels", "median"),
        median_decode_ms=("decode_ms", "median"),
        median_preprocess_ms=("model_preprocess_ms", "median"),
        median_inference_ms=("inference_ms", "median"),
        p95_inference_ms=("inference_ms", p95),
        median_output_kb=("output_kb", "median"),
        median_estimated_full_inference_ms=(
            "estimated_full_inference_ms",
            "median"
        ),
    )
)

display(summary)

,workload,model,device,samples,median_input_mb,median_megapixels,median_decode_ms,median_preprocess_ms,median_inference_ms,p95_inference_ms,median_output_kb,median_estimated_full_inference_ms
0,Leafy context classification,MobileNetV3-Small,cpu,20,4.000870,1.048576,5.608042,3.580812,48.767854,56.694313,3.90625,48.767854
1,Leafy crop classification,MobileNetV3-Small,cpu,20,0.006172,0.001521,1.654959,0.824896,47.122084,51.651300,3.90625,47.122084
2,Leafy orthomosaic tiled analysis,MobileNetV3-Small tiled proxy,cpu,4,26.450594,16.777216,313.064708,8.511271,246.924354,319.121540,15.62500,2791.581548
3,Rumex weed detection,FasterRCNN-MobileNetV3-320-FPN,cpu,8,5.203753,2.304000,59.755375,4.643250,112.261708,129.435402,0.00000,112.261708


## 15. Add simple network-transfer estimates

These remain idealized transfer times and are only used to connect Stage 3 profiles to the future NOMS simulator.

In [19]:
for bw in [5, 10, 25, 50, 100]:
    profile_df[f"upload_sec_{bw}Mbps"] = (
        profile_df["input_size_mb"] * 8 / bw
    )

transfer_cols = [
    "workload",
    "input_size_mb",
    "inference_ms",
    "estimated_full_inference_ms",
    "upload_sec_5Mbps",
    "upload_sec_10Mbps",
    "upload_sec_25Mbps",
    "upload_sec_50Mbps",
    "upload_sec_100Mbps",
]

display(profile_df[transfer_cols].head(20))

,workload,input_size_mb,inference_ms,estimated_full_inference_ms,upload_sec_5Mbps,upload_sec_10Mbps,upload_sec_25Mbps,upload_sec_50Mbps,upload_sec_100Mbps
0,Leafy crop classification,0.006172,63.927834,63.927834,0.009875,0.004938,0.001975,0.000988,0.000494
1,Leafy crop classification,0.006172,51.005167,51.005167,0.009875,0.004938,0.001975,0.000988,0.000494
2,Leafy crop classification,0.006172,49.624958,49.624958,0.009875,0.004938,0.001975,0.000988,0.000494
3,Leafy crop classification,0.006172,47.724042,47.724042,0.009875,0.004938,0.001975,0.000988,0.000494
4,Leafy crop classification,0.006172,46.957459,46.957459,0.009875,0.004938,0.001975,0.000988,0.000494
5,Leafy crop classification,0.006172,46.455375,46.455375,0.009875,0.004938,0.001975,0.000988,0.000494
6,Leafy crop classification,0.006172,46.969542,46.969542,0.009875,0.004938,0.001975,0.000988,0.000494
7,Leafy crop classification,0.006172,46.706875,46.706875,0.009875,0.004938,0.001975,0.000988,0.000494
8,Leafy crop classification,0.006172,47.987542,47.987542,0.009875,0.004938,0.001975,0.000988,0.000494
9,Leafy crop classification,0.006172,47.219416,47.219416,0.009875,0.004938,0.001975,0.000988,0.000494


## 16. Save Stage 3 outputs

In [20]:
raw_path = OUTPUT_DIR / "stage3_model_profile_raw.csv"
summary_path = OUTPUT_DIR / "stage3_workload_profile_summary.csv"

profile_df.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)

print("Saved:")
print(" -", raw_path)
print(" -", summary_path)

Saved:
 - /Users/bwbgv/Downloads/NOMS_Fieldvision/eda_outputs/stage3_model_profile_raw.csv
 - /Users/bwbgv/Downloads/NOMS_Fieldvision/eda_outputs/stage3_workload_profile_summary.csv


## 17. How to interpret these numbers

For the NOMS paper, the most useful fields are:

- `input_size_mb` → network demand
- `decode_ms` → input handling cost
- `model_preprocess_ms` → preprocessing cost
- `inference_ms` → measured compute latency
- `estimated_full_inference_ms` → **only** for the lightweight orthomosaic estimate
- `output_kb` → approximate return-payload size
- `extra_peak_cuda_mb` → optional GPU-memory pressure
- `device` → hardware on which this trace was collected

### Important

These models are **proxy workloads** for systems profiling.

They are not yet the final SenSys agricultural models, and their prediction accuracy should not be reported as an agricultural result.

Once we select the final frozen perception models, we can rerun this exact profiler on those models without changing the RL methodology.